# Strava export → Feature Contract v0

Converts a Strava `activities.csv` export into the 28 model input columns defined in [`FEATURE_CONTRACT.md`](../FEATURE_CONTRACT.md).

The code lives in one module; this notebook only runs them and shows examples:

- [`RunGuard/ml_logic/strava.py`](../RunGuard/ml_logic/strava.py): `clean_activities(path)`, the clean activity log (task A1), and `build_features(activities, event_day)`, one row of contract columns (task A2)

Conversion rules follow [`docs/STRAVA_CONVERSION_DECISIONS.md`](../docs/STRAVA_CONVERSION_DECISIONS.md):

1. `Activity Date` is kept in UTC.
2. Runs shorter than 2 km (mostly separately recorded warm-ups and cool-downs) are removed completely.
3. Distances come from `Distance.1` (always metres), converted to kilometres. The other `Distance` column is ignored.
4. Activities that run past midnight are spread evenly over the time they cover (elapsed time, or moving time if the recording was left running).

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

# Make the repo root importable, so `import RunGuard` finds the package next to notebooks/.
# Not needed once the package is installed with `pip install -e .`
sys.path.insert(0, str(Path.cwd().parent))

from RunGuard.ml_logic.strava import (
    MAX_SESSIONS,
    MIN_RUN_KM,
    RUN_TYPES,
    clean_activities,
    build_features,
    contract_rows,
    daily_features,
)

STRAVA_CSV = Path("../raw_data/dominik_strava_data_full_export_raw.csv")  # raw_data/ is git-ignored
CONTRACT_MD = Path("../FEATURE_CONTRACT.md")

## Clean activity log

In [ ]:
activities = clean_activities(STRAVA_CSV)

# For the summary only: compare with the raw export
raw = pd.read_csv(STRAVA_CSV, low_memory=False)
raw_runs = raw["Activity Type"].isin(RUN_TYPES)
short_runs = raw_runs & (raw["Distance.1"] / 1000 < MIN_RUN_KM)

print(f"{len(raw)} activities in the export, {len(activities)} used")
print(f"short runs removed: {short_runs.sum()} of {raw_runs.sum()} runs, "
      f"{raw.loc[short_runs, 'Distance.1'].sum() / 1000:.0f} km")
print(activities["group"].value_counts().to_dict())
activities.head()

## Daily features

In [ ]:
daily = daily_features(activities)
print(f"{len(daily)} calendar days, {daily.index[0].date()} to {daily.index[-1].date()}")

# 9 Aug 2025: warm-up (0.81 km) and cool-down (1.78 km) are removed, the 5.02 km parkrun stays
daily.loc["2025-08-07":"2025-08-11"]

In [ ]:
# Paris-Brest-Paris: 55.9 moving hours over 89.7 elapsed hours from 2019-08-18 17:28 UTC,
# spread over the 5 days it covers (about 15 hours and 326 km on each full day)
daily.loc["2019-08-17":"2019-08-23"]

## Model input

`build_features` returns the row the model reads. By default the event day is the day after the last activity.

In [ ]:
row = build_features(activities)
print(row.shape)
row

In [ ]:
# Any other event day, e.g. the day after the 9 Aug 2025 parkrun: total km.6 = 5.02
build_features(activities, "2025-08-10")

## Full history

One row for every day with a complete 7-day history, useful for exploring the data.

In [ ]:
rows = contract_rows(daily)
print(rows.shape)
rows.head()

## Checks

In [ ]:
# Column names and order are copied from the contract, not retyped
block = re.search(r"```\n(nr\. sessions\n.*?)```", CONTRACT_MD.read_text(), re.S).group(1)
contract_order = [line for line in block.splitlines() if line]
assert list(row.columns) == contract_order, "column names or order differ from FEATURE_CONTRACT.md"
assert list(rows.columns) == contract_order

assert rows.notna().all().all()
assert rows.filter(like="nr. sessions").max().max() <= MAX_SESSIONS

# build_features gives the same row as the full history for the same event day
assert row.equals(rows.tail(1))

# Decision 2: no run shorter than MIN_RUN_KM is left
assert (activities.loc[activities["group"] == "run", "km"] >= MIN_RUN_KM).all()

# Splitting by day (decision 4) neither loses nor adds kilometres or hours (daily values are rounded)
run_km = activities.loc[activities["group"] == "run", "km"].sum()
assert abs(daily["total km"].sum() - run_km) < 1, (daily["total km"].sum(), run_km)

alternative_hours = activities.loc[activities["group"] == "alternative", "moving_s"].sum() / 3600
assert abs(daily["hours alternative"].sum() - alternative_hours) < 1, (daily["hours alternative"].sum(), alternative_hours)

# No day can hold more than 24 hours
assert daily["hours alternative"].max() <= 24

print(f"max hours alternative per day: {daily['hours alternative'].max()}")
print("all checks passed")

## Save

Runs only after the checks above have passed.

In [ ]:
# Save the results next to the export. raw_data/ is git-ignored: these files hold personal training data.
OUTPUT_DIR = STRAVA_CSV.parent
model_input_csv = OUTPUT_DIR / f"model_input_{row.index[0].date()}.csv"
history_csv = OUTPUT_DIR / "contract_rows_full_history.csv"

# The index ("event day") is written as the first column; the 28 contract columns follow in contract order
row.to_csv(model_input_csv)
rows.to_csv(history_csv)

# Read the file back to make sure it holds exactly what we computed
saved = pd.read_csv(model_input_csv, index_col="event day", parse_dates=["event day"])
assert list(saved.columns) == list(row.columns)
assert (saved.values == row.values).all()

print(f"saved {model_input_csv} ({row.shape[0]} row)")
print(f"saved {history_csv} ({rows.shape[0]} rows)")